# Notebook 05 — Feature Engineering & Preprocessing

## Objective

Transform the Train, Validation, and Test datasets into
model-ready features.

## Prediction Point

**Order approval time**

Only features available at or before `order_approved_at`
are eligible.

## Main Steps

1. Create engineered features.
2. Remove leakage and identifiers.
3. Handle missing values.
4. Encode categorical variables.
5. Transform skewed numerical features.
6. Scale numerical features.
7. Fit transformations on training data only.
8. Apply them unchanged to Validation and Test.

## Output

Model-ready feature tables and fitted preprocessing artifacts.

In [101]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer

In [102]:
TRAIN_PATH = Path("../artifacts/notebook_03/train.parquet")
VALIDATION_PATH = Path("../artifacts/notebook_03/validation.parquet")
TEST_PATH = Path("../artifacts/notebook_03/test.parquet")

OUTPUT_DIR = Path("../artifacts/notebook_05")
TRANSFORMERS_DIR = OUTPUT_DIR / "transformers"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRANSFORMERS_DIR.mkdir(parents=True, exist_ok=True)

print("Train:", TRAIN_PATH)
print("Validation:", VALIDATION_PATH)
print("Test:", TEST_PATH)
print("Output:", OUTPUT_DIR)
print("Transformers:", TRANSFORMERS_DIR)

Train: ..\artifacts\notebook_03\train.parquet
Validation: ..\artifacts\notebook_03\validation.parquet
Test: ..\artifacts\notebook_03\test.parquet
Output: ..\artifacts\notebook_05
Transformers: ..\artifacts\notebook_05\transformers


In [103]:
train = pd.read_parquet(TRAIN_PATH)
validation = pd.read_parquet(VALIDATION_PATH)
test = pd.read_parquet(TEST_PATH)

print("Train shape:", train.shape)
print("Validation shape:", validation.shape)
print("Test shape:", test.shape)

Train shape: (67532, 22)
Validation shape: (14472, 22)
Test shape: (14472, 22)


In [104]:
assert train["order_id"].is_unique
assert validation["order_id"].is_unique
assert test["order_id"].is_unique

assert set(train["order_id"]).isdisjoint(validation["order_id"])
assert set(train["order_id"]).isdisjoint(test["order_id"])
assert set(validation["order_id"]).isdisjoint(test["order_id"])

assert train["late"].notna().all()
assert validation["late"].notna().all()
assert test["late"].notna().all()

print("Split integrity checks passed.")

Split integrity checks passed.


In [105]:
PREDICTION_POINT = "order_approval"

TARGET_COLUMN = "late"

ID_COLUMNS = [
    "order_id",
    "customer_id",
]

LEAKAGE_COLUMNS = [
    "delivery_delay_days",
    "label_data_available",
    "order_delivered_customer_date",
    "order_delivered_carrier_date",
    "order_status",
]

RAW_DATE_COLUMNS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_estimated_delivery_date",
]

print("Prediction point:", PREDICTION_POINT)
print("Target:", TARGET_COLUMN)
print("Excluded identifiers:", ID_COLUMNS)
print("Excluded leakage/post-outcome columns:", LEAKAGE_COLUMNS)

Prediction point: order_approval
Target: late
Excluded identifiers: ['order_id', 'customer_id']
Excluded leakage/post-outcome columns: ['delivery_delay_days', 'label_data_available', 'order_delivered_customer_date', 'order_delivered_carrier_date', 'order_status']


In [106]:
assert TARGET_COLUMN not in LEAKAGE_COLUMNS

for df_name, df in {
    "train": train,
    "validation": validation,
    "test": test
}.items():

    required_columns = (
        LEAKAGE_COLUMNS
        + ID_COLUMNS
        + [
            "order_purchase_timestamp",
            "order_approved_at",
            "order_estimated_delivery_date",
        ]
    )

    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    assert not missing_columns, (
        f"{df_name}: required columns are missing: {missing_columns}"
    )

print("Raw input schema validation passed.")

Raw input schema validation passed.


## Feature Engineering Strategy

Features are based on the EDA findings and the prediction-time contract.

### Prediction Point

**Order approval time**

### Included Order Aggregates

- `item_count`
- `unique_products`
- `unique_sellers`
- `total_price`
- `total_freight_value`
- `payment_count`
- `payment_types_count`

The availability of payment aggregates at the prediction point
is a documented business-time assumption and should be verified
before production use.

### Principles

- Preserve one row per order.
- Exclude leakage and post-prediction information.
- Fit learned transformations on training data only.
- Reuse fitted transformations for validation, test, and inference.

In [107]:
def create_features(df):
    out = pd.DataFrame(index=df.index)

    purchase = pd.to_datetime(
        df["order_purchase_timestamp"],
        errors="coerce"
    )

    approved = pd.to_datetime(
        df["order_approved_at"],
        errors="coerce"
    )

    estimated = pd.to_datetime(
        df["order_estimated_delivery_date"],
        errors="coerce"
    )

    # -------------------------
    # Temporal features
    # -------------------------

    out["purchase_year"] = purchase.dt.year

    purchase_month = purchase.dt.month
    out["purchase_month_sin"] = np.sin(
        2 * np.pi * purchase_month / 12
    )
    out["purchase_month_cos"] = np.cos(
        2 * np.pi * purchase_month / 12
    )

    purchase_dow = purchase.dt.dayofweek
    out["purchase_dow_sin"] = np.sin(
        2 * np.pi * purchase_dow / 7
    )
    out["purchase_dow_cos"] = np.cos(
        2 * np.pi * purchase_dow / 7
    )

    purchase_hour = purchase.dt.hour
    out["purchase_hour_sin"] = np.sin(
        2 * np.pi * purchase_hour / 24
    )
    out["purchase_hour_cos"] = np.cos(
        2 * np.pi * purchase_hour / 24
    )

    day_of_week = purchase.dt.dayofweek
    out["is_weekend"] = np.where(
        day_of_week.isna(),
        np.nan,
        (day_of_week >= 5).astype(int)
    )
    # -------------------------
    # Approval timing
    # -------------------------

    approval_delay = (
        approved - purchase
    ).dt.total_seconds() / 3600

    out["approval_delay_hours"] = approval_delay
    out["approval_delay_missing"] = approval_delay.isna().astype(int)

    # -------------------------
    # Estimated delivery window
    # -------------------------

    estimated_lead = (
        estimated - purchase
    ).dt.total_seconds() / 86400

    out["estimated_delivery_lead_days"] = estimated_lead

    # -------------------------
    # Order structure
    # -------------------------

    out["item_count"] = df["item_count"]
    out["unique_products"] = df["unique_products"]
    out["unique_sellers"] = df["unique_sellers"]

    # -------------------------
    # Monetary / shipping
    # -------------------------

    out["total_price"] = df["total_price"]
    out["total_freight_value"] = df["total_freight_value"]

    out["freight_ratio"] = np.where(
        df["total_price"] > 0,
        df["total_freight_value"] / df["total_price"],
        0.0
    )

    # -------------------------
    # Payment
    # -------------------------

    out["payment_count"] = df["payment_count"]
    out["payment_types_count"] = df["payment_types_count"]

    # -------------------------
    # Geography
    # -------------------------

    out["customer_state"] = df["customer_state"]
    out["customer_city"] = df["customer_city"]
    out["customer_zip_code_prefix"] = (
        df["customer_zip_code_prefix"].astype("string")
    )

    return out

In [108]:
X_train_raw = create_features(train)
X_validation_raw = create_features(validation)
X_test_raw = create_features(test)

y_train = train[TARGET_COLUMN].copy()
y_validation = validation[TARGET_COLUMN].copy()
y_test = test[TARGET_COLUMN].copy()

print("Raw engineered train shape:", X_train_raw.shape)
print("Raw engineered validation shape:", X_validation_raw.shape)
print("Raw engineered test shape:", X_test_raw.shape)

Raw engineered train shape: (67532, 22)
Raw engineered validation shape: (14472, 22)
Raw engineered test shape: (14472, 22)


In [109]:
for split_name, X in {
    "train": X_train_raw,
    "validation": X_validation_raw,
    "test": X_test_raw,
}.items():

    assert X["approval_delay_missing"].isin([0, 1]).all(), (
        f"{split_name}: approval_delay_missing contains invalid values."
    )

    assert (
        X["estimated_delivery_lead_days"] > 0
    ).all(), (
        f"{split_name}: estimated_delivery_lead_days contains "
        "non-positive values."
    )

    assert (
        X["freight_ratio"] >= 0
    ).all(), (
        f"{split_name}: freight_ratio contains negative values."
    )

print("Engineered feature validation passed for all splits.")

Engineered feature validation passed for all splits.


In [110]:
RAW_FORBIDDEN_FEATURES = set(
    ID_COLUMNS
    + LEAKAGE_COLUMNS
    + [TARGET_COLUMN]
)

for split_name, X in {
    "train": X_train_raw,
    "validation": X_validation_raw,
    "test": X_test_raw,
}.items():

    leaked_columns = (
        RAW_FORBIDDEN_FEATURES
        & set(X.columns)
    )

    assert not leaked_columns, (
        f"{split_name}: forbidden columns leaked into "
        f"engineered features: {sorted(leaked_columns)}"
    )

print("Raw engineered feature leakage audit passed.")

Raw engineered feature leakage audit passed.


In [111]:
missing_after_engineering = (
    X_train_raw.isna()
    .sum()
    .sort_values(ascending=False)
)

display(
    missing_after_engineering[
        missing_after_engineering > 0
    ].to_frame("missing_count")
)

,missing_count
approval_delay_hours,12


In [112]:
class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.mapping_ = None
        self.default_ = 0.0

    def fit(self, X, y=None):
        X_series = pd.Series(X).astype("string")

        frequencies = (
            X_series
            .value_counts(normalize=True, dropna=False)
        )

        self.mapping_ = frequencies.to_dict()

        return self

    def transform(self, X):
        if self.mapping_ is None:
            raise RuntimeError(
                "FrequencyEncoder must be fitted before transform."
            )

        X_series = pd.Series(X).astype("string")

        encoded = (
            X_series
            .map(self.mapping_)
            .fillna(self.default_)
            .astype(float)
        )

        return encoded.to_numpy().reshape(-1, 1)

In [113]:
city_frequency_encoder = FrequencyEncoder()

city_frequency_encoder.fit(
    X_train_raw["customer_city"]
)

joblib.dump(
    city_frequency_encoder,
    TRANSFORMERS_DIR / "city_frequency_encoder.joblib"
)

print(
    "City frequency encoder fitted on training data."
)
print(
    "Number of learned city categories:",
    len(city_frequency_encoder.mapping_)
)

City frequency encoder fitted on training data.
Number of learned city categories: 3668


In [114]:
zip_frequency_encoder = FrequencyEncoder()

zip_frequency_encoder.fit(
    X_train_raw["customer_zip_code_prefix"]
)

joblib.dump(
    zip_frequency_encoder,
    TRANSFORMERS_DIR / "zip_frequency_encoder.joblib"
)

print(
    "ZIP frequency encoder fitted on training data."
)
print(
    "Number of learned ZIP categories:",
    len(zip_frequency_encoder.mapping_)
)

ZIP frequency encoder fitted on training data.
Number of learned ZIP categories: 13709


In [115]:
def apply_frequency_features(
    X,
    city_encoder,
    zip_encoder
):
    out = X.copy()

    out["city_frequency"] = (
        city_encoder
        .transform(out["customer_city"])
        .ravel()
    )

    out["zip_frequency"] = (
        zip_encoder
        .transform(out["customer_zip_code_prefix"])
        .ravel()
    )

    out = out.drop(
        columns=[
            "customer_city",
            "customer_zip_code_prefix"
        ]
    )

    return out

In [116]:
X_train_prepared = apply_frequency_features(
    X_train_raw,
    city_frequency_encoder,
    zip_frequency_encoder
)

X_validation_prepared = apply_frequency_features(
    X_validation_raw,
    city_frequency_encoder,
    zip_frequency_encoder
)

X_test_prepared = apply_frequency_features(
    X_test_raw,
    city_frequency_encoder,
    zip_frequency_encoder
)

print("Train:", X_train_prepared.shape)
print("Validation:", X_validation_prepared.shape)
print("Test:", X_test_prepared.shape)

Train: (67532, 22)
Validation: (14472, 22)
Test: (14472, 22)


## Numerical Transformation

Right-skewed non-negative features are transformed using `log1p`
before scaling.

Features include:

- `item_count`
- `unique_products`
- `unique_sellers`
- `total_price`
- `total_freight_value`
- `payment_count`
- `payment_types_count`
- `freight_ratio`
- `city_frequency`
- `zip_frequency`

Outliers are retained because extreme values may represent
valid e-commerce orders.

In [117]:
LOG_NUMERICAL_FEATURES = [
    "item_count",
    "unique_products",
    "unique_sellers",
    "total_price",
    "total_freight_value",
    "payment_count",
    "payment_types_count",
    "freight_ratio",
    "city_frequency",
    "zip_frequency",
]

REGULAR_NUMERICAL_FEATURES = [
    "approval_delay_hours",
    "estimated_delivery_lead_days",
    "purchase_month_sin",
    "purchase_month_cos",
    "purchase_dow_sin",
    "purchase_dow_cos",
    "purchase_hour_sin",
    "purchase_hour_cos",
]

CATEGORICAL_FEATURES = [
    "purchase_year",
    "customer_state",
]

BINARY_FEATURES = [
    "approval_delay_missing",
    "is_weekend",
]

print("Log numerical:", LOG_NUMERICAL_FEATURES)
print("Regular numerical:", REGULAR_NUMERICAL_FEATURES)
print("Categorical:", CATEGORICAL_FEATURES)
print("Binary:", BINARY_FEATURES)

Log numerical: ['item_count', 'unique_products', 'unique_sellers', 'total_price', 'total_freight_value', 'payment_count', 'payment_types_count', 'freight_ratio', 'city_frequency', 'zip_frequency']
Regular numerical: ['approval_delay_hours', 'estimated_delivery_lead_days', 'purchase_month_sin', 'purchase_month_cos', 'purchase_dow_sin', 'purchase_dow_cos', 'purchase_hour_sin', 'purchase_hour_cos']
Categorical: ['purchase_year', 'customer_state']
Binary: ['approval_delay_missing', 'is_weekend']


In [118]:
all_model_input_features = (
    LOG_NUMERICAL_FEATURES
    + REGULAR_NUMERICAL_FEATURES
    + CATEGORICAL_FEATURES
    + BINARY_FEATURES
)

missing_features = [
    col
    for col in all_model_input_features
    if col not in X_train_prepared.columns
]

assert not missing_features, (
    f"Missing model input features: {missing_features}"
)

assert len(all_model_input_features) == len(
    set(all_model_input_features)
)

print(
    "Feature group validation passed."
)
print(
    "Total preprocessed input features:",
    len(all_model_input_features)
)

Feature group validation passed.
Total preprocessed input features: 22


In [119]:
for split_name, X in {
    "train": X_train_prepared,
    "validation": X_validation_prepared,
    "test": X_test_prepared,
}.items():

    for feature in LOG_NUMERICAL_FEATURES:

        min_value = X[feature].min(skipna=True)

        assert min_value >= 0, (
            f"{split_name}: {feature} has negative value "
            f"{min_value}, invalid for log1p transformation."
        )

print("All log-transformed features are valid for log1p.")

All log-transformed features are valid for log1p.


In [120]:
log_numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "log1p",
            FunctionTransformer(
                np.log1p,
                feature_names_out="one-to-one"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

regular_numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)

binary_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "log_numeric",
            log_numeric_pipeline,
            LOG_NUMERICAL_FEATURES
        ),
        (
            "regular_numeric",
            regular_numeric_pipeline,
            REGULAR_NUMERICAL_FEATURES
        ),
        (
            "categorical",
            categorical_pipeline,
            CATEGORICAL_FEATURES
        ),
        (
            "binary",
            binary_pipeline,
            BINARY_FEATURES
        ),
    ],
    remainder="drop"
)

In [121]:
preprocessor.fit(X_train_prepared)

print(
    "Preprocessor fitted using TRAIN ONLY."
)

Preprocessor fitted using TRAIN ONLY.


In [122]:
X_train_final = preprocessor.transform(
    X_train_prepared
)

X_validation_final = preprocessor.transform(
    X_validation_prepared
)

X_test_final = preprocessor.transform(
    X_test_prepared
)

print("Final train matrix:", X_train_final.shape)
print("Final validation matrix:", X_validation_final.shape)
print("Final test matrix:", X_test_final.shape)

Final train matrix: (67532, 50)
Final validation matrix: (14472, 50)
Final test matrix: (14472, 50)


In [123]:
assert X_train_final.shape[1] == X_validation_final.shape[1]
assert X_train_final.shape[1] == X_test_final.shape[1]

assert X_train_final.shape[0] == len(train)
assert X_validation_final.shape[0] == len(validation)
assert X_test_final.shape[0] == len(test)

print("Final feature matrix consistency checks passed.")

Final feature matrix consistency checks passed.


In [124]:
feature_names = (
    preprocessor
    .get_feature_names_out()
    .tolist()
)

print(
    "Number of final features:",
    len(feature_names)
)

display(
    pd.DataFrame({
        "feature_index": range(len(feature_names)),
        "feature_name": feature_names
    }).head(30)
)

Number of final features: 50


,feature_index,feature_name
0,0,log_numeric__item_count
1,1,log_numeric__unique_products
2,2,log_numeric__unique_sellers
3,3,log_numeric__total_price
4,4,log_numeric__total_freight_value
5,5,log_numeric__payment_count
6,6,log_numeric__payment_types_count
7,7,log_numeric__freight_ratio
8,8,log_numeric__city_frequency
9,9,log_numeric__zip_frequency


In [125]:
feature_list = pd.DataFrame({
    "feature_index": range(len(feature_names)),
    "feature_name": feature_names
})

feature_list.to_csv(
    OUTPUT_DIR / "feature_list.csv",
    index=False
)

with open(
    OUTPUT_DIR / "feature_list.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        feature_names,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    "Feature list saved."
)

Feature list saved.


In [126]:
joblib.dump(
    preprocessor,
    TRANSFORMERS_DIR / "preprocessor.joblib"
)

print(
    "Fitted preprocessing pipeline saved."
)

Fitted preprocessing pipeline saved.


In [127]:
joblib.dump(
    preprocessor.named_transformers_["log_numeric"],
    TRANSFORMERS_DIR / "log_numeric_pipeline.joblib"
)

joblib.dump(
    preprocessor.named_transformers_["regular_numeric"],
    TRANSFORMERS_DIR / "regular_numeric_pipeline.joblib"
)

joblib.dump(
    preprocessor.named_transformers_["categorical"],
    TRANSFORMERS_DIR / "categorical_pipeline.joblib"
)

joblib.dump(
    preprocessor.named_transformers_["binary"],
    TRANSFORMERS_DIR / "binary_pipeline.joblib"
)

print("Individual fitted preprocessing objects saved.")

Individual fitted preprocessing objects saved.


In [128]:
from scipy import sparse

In [129]:
# Save final feature matrices as Parquet files

X_train_final_df = pd.DataFrame(
    X_train_final,
    columns=feature_names
)

X_validation_final_df = pd.DataFrame(
    X_validation_final,
    columns=feature_names
)

X_test_final_df = pd.DataFrame(
    X_test_final,
    columns=feature_names
)

X_train_final_df.to_parquet(
    OUTPUT_DIR / "X_train.parquet",
    index=False
)

X_validation_final_df.to_parquet(
    OUTPUT_DIR / "X_validation.parquet",
    index=False
)

X_test_final_df.to_parquet(
    OUTPUT_DIR / "X_test.parquet",
    index=False
)

y_train.to_frame("late").to_parquet(
    OUTPUT_DIR / "y_train.parquet",
    index=False
)

y_validation.to_frame("late").to_parquet(
    OUTPUT_DIR / "y_validation.parquet",
    index=False
)

y_test.to_frame("late").to_parquet(
    OUTPUT_DIR / "y_test.parquet",
    index=False
)

print("Final feature matrices and targets saved successfully.")

Final feature matrices and targets saved successfully.


In [130]:
train[["order_id"]].to_parquet(
    OUTPUT_DIR / "train_order_ids.parquet",
    index=False
)

validation[["order_id"]].to_parquet(
    OUTPUT_DIR / "validation_order_ids.parquet",
    index=False
)

test[["order_id"]].to_parquet(
    OUTPUT_DIR / "test_order_ids.parquet",
    index=False
)

print("Order identifiers saved separately for traceability.")

Order identifiers saved separately for traceability.


In [131]:
FORBIDDEN_FEATURE_TOKENS = [
    "late",
    "delivery_delay_days",
    "label_data_available",
    "order_delivered_customer_date",
    "order_delivered_carrier_date",
    "order_status",
    "order_id",
    "customer_id",
]

leaked_final_features = [
    feature_name
    for feature_name in feature_names
    if any(
        token in feature_name
        for token in FORBIDDEN_FEATURE_TOKENS
    )
]

assert not leaked_final_features, (
    "Forbidden information detected in final features: "
    f"{leaked_final_features}"
)

print("Final feature leakage audit passed.")

Final feature leakage audit passed.


In [132]:
assert np.isfinite(X_train_final).all()
assert np.isfinite(X_validation_final).all()
assert np.isfinite(X_test_final).all()

print(
    "All final feature matrices contain only finite values."
)

All final feature matrices contain only finite values.


In [133]:
assert len(y_train) == X_train_final.shape[0]
assert len(y_validation) == X_validation_final.shape[0]
assert len(y_test) == X_test_final.shape[0]

assert y_train.index.equals(train.index)
assert y_validation.index.equals(validation.index)
assert y_test.index.equals(test.index)

print("Target-feature alignment checks passed.")

Target-feature alignment checks passed.


In [134]:
feature_spec = {
    "prediction_point": PREDICTION_POINT,
    "target": TARGET_COLUMN,

    "excluded_columns": (
        ID_COLUMNS
        + LEAKAGE_COLUMNS
    ),

    "log_numeric_features": LOG_NUMERICAL_FEATURES,

    "regular_numeric_features": (
        REGULAR_NUMERICAL_FEATURES
    ),

    "categorical_features": (
        CATEGORICAL_FEATURES
    ),

    "binary_features": (
        BINARY_FEATURES
    ),

    "frequency_encoded_features": {
        "customer_city": "city_frequency",
        "customer_zip_code_prefix": "zip_frequency"
    },

    "excluded_redundant_feature": {
        "feature": "total_payment_value",
        "reason": (
            "Excluded from the primary feature set because "
            "it showed very high correlation with total_price "
            "in Notebook 04. It may be evaluated later through "
            "an ablation experiment."
        )
    },

    "prediction_time_assumptions": {
        "order_aggregates": [
            "item_count",
            "unique_products",
            "unique_sellers",
            "total_price",
            "total_freight_value"
        ],
        "payment_aggregates": [
            "payment_count",
            "payment_types_count"
        ],
        "note": (
            "Payment and order-level aggregate availability at "
            "order approval is a business-time assumption and "
            "must be verified before production deployment."
        )
    },

    "scaling": (
        "StandardScaler for numerical features"
    ),

    "categorical_encoding": (
        "OneHotEncoder(handle_unknown='ignore')"
    ),

    "high_cardinality_encoding": (
        "Frequency encoding fitted on training data only"
    ),

    "numeric_transformation": (
        "log1p for strongly right-skewed non-negative features"
    ),

    "missing_value_strategy": (
        "Median imputation for numerical features and "
        "most-frequent imputation for categorical/binary features"
    ),

    "fit_rule": (
        "All fitted transformations are learned from training "
        "data only and reused unchanged for validation, test, "
        "and production inference."
    )
}

with open(
    OUTPUT_DIR / "feature_spec.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        feature_spec,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Feature specification saved.")

Feature specification saved.


In [135]:
loaded_preprocessor = joblib.load(
    TRANSFORMERS_DIR / "preprocessor.joblib"
)

loaded_city_encoder = joblib.load(
    TRANSFORMERS_DIR / "city_frequency_encoder.joblib"
)

loaded_zip_encoder = joblib.load(
    TRANSFORMERS_DIR / "zip_frequency_encoder.joblib"
)

print("Saved preprocessing objects loaded successfully.")

Saved preprocessing objects loaded successfully.


In [136]:
X_validation_reloaded = apply_frequency_features(
    X_validation_raw,
    loaded_city_encoder,
    loaded_zip_encoder
)

X_validation_reloaded_final = (
    loaded_preprocessor.transform(
        X_validation_reloaded
    )
)

assert (
    X_validation_reloaded_final.shape
    == X_validation_final.shape
)

print(
    "Production-style reload and transformation test passed."
)

Production-style reload and transformation test passed.


In [137]:
artifacts = sorted(
    str(path.relative_to(OUTPUT_DIR))
    for path in OUTPUT_DIR.rglob("*")
    if path.is_file()
)

artifact_manifest = pd.DataFrame({
    "artifact": artifacts
})

artifact_manifest.to_csv(
    OUTPUT_DIR / "artifact_manifest.csv",
    index=False
)

display(artifact_manifest)

,artifact
0,X_test.parquet
1,X_train.parquet
2,X_validation.parquet
3,artifact_manifest.csv
4,feature_list.csv
5,feature_list.json
6,feature_spec.json
7,notebook_05_summary.json
8,test_order_ids.parquet
9,train_order_ids.parquet


In [138]:
summary = {
    "train_rows": X_train_final.shape[0],
    "validation_rows": X_validation_final.shape[0],
    "test_rows": X_test_final.shape[0],

    "final_feature_count": X_train_final.shape[1],

    "target_train_late_rate": float(
        y_train.mean()
    ),

    "target_validation_late_rate": float(
        y_validation.mean()
    ),

    "target_test_late_rate": float(
        y_test.mean()
    ),

    "prediction_point": PREDICTION_POINT,

    "preprocessing_fit_on": (
        "training split only"
    ),

    "feature_encoding": {
        "customer_city": "frequency encoding",
        "customer_zip_code_prefix": "frequency encoding",
        "customer_state": "one-hot encoding",
        "purchase_year": "one-hot encoding"
    },

    "scaling": "StandardScaler",

    "skew_handling": "log1p",

    "payment_features": [
        "payment_count",
        "payment_types_count"
    ],

    "payment_availability": (
        "business-time assumption requiring verification"
    ),

    "leakage_columns_excluded": LEAKAGE_COLUMNS,

    "identifiers_excluded": ID_COLUMNS
}

display(
    pd.DataFrame(
        summary.items(),
        columns=["metric", "value"]
    )
)

with open(
    OUTPUT_DIR / "notebook_05_summary.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Notebook 05 summary saved.")

,metric,value
0,train_rows,67532
1,validation_rows,14472
2,test_rows,14472
3,final_feature_count,50
4,target_train_late_rate,0.081132
5,target_validation_late_rate,0.081122
6,target_test_late_rate,0.081122
7,prediction_point,order_approval
8,preprocessing_fit_on,training split only
9,feature_encoding,"{'customer_city': 'frequency encoding', 'custo..."


Notebook 05 summary saved.
